# LHRM — Client Inference Notebook | Frozen R_AUTHOR Epoch 6

**Input:** Your own English scientific abstract. **Output:** Top-5 ranked raw AAPD label codes and sigmoid scores.

**This is inference only.** No training, no optimizer, no Train/Dev/Test dataset needed, no HRRM, no modifications to the checkpoint.

### Kaggle quick start

1. Create a new Kaggle notebook by importing this file.
2. **Add Input:** the existing **`lhrm-gpu-bundle-r-author-20261015`** code dataset (16 verified files).
3. **Add Input:** the **private** `lhrm-r-author-epoch6-test-artifacts` dataset containing **`best_dev_model_only.pt`** and **`EVIDENCE.zip`** (Kaggle may auto-extract the ZIP).
4. In Notebook Settings select a **GPU** if available (CPU works but is slower); enable Internet for the first download of the pinned BERT tokenizer/config if those files are not already cached.
5. Execute **Cells 1–4 in order**, then use **Cell 5** (edit a Python string) or **Cell 6** (interactive text box) to enter abstracts and receive predictions.

### Important scientific caveats

- Model: **R_AUTHOR + BCE, trained on AAPD Train**; checkpoint selected on **Dev** at **Epoch 6 / Step 40,380**, no Test tuning.
- The output label codes are the **original 54 AAPD CSV columns**. The real-world **semantic mapping of some labels is unresolved**. Do not interpret a seemingly strange label as verified subject metadata; do not silently rename labels.
- **Sigmoid scores are not calibrated probabilities.** They are model outputs for ranking; this is not a clinical or other high-stakes classification system.
- Long inputs are truncated to **512 BERT tokens**. This is an English-language BERT model.
- **Privacy:** If you save/share the notebook with cell outputs, your submitted abstracts and predictions may be included. Clear outputs before sharing.

**Immutable provenance:** `AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A`, original AAPD label hash `a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b`, selected checkpoint SHA-256 `988bb64fca2b8ea21030585de7268a1cfb1fc8676b91b4d6e6f38c4f22c95643`.


In [ ]:
# CELL 1 — CLIENT PREFLIGHT: verify frozen provenance; does not read Train/Dev/Test contents.
from pathlib import Path
from io import BytesIO
import os, sys, json, hashlib, zipfile, shutil, random, time
import numpy as np
import pandas as pd
import torch

RUN_ID = "AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A"
EXPECTED_CHECKPOINT_SHA = '988bb64fca2b8ea21030585de7268a1cfb1fc8676b91b4d6e6f38c4f22c95643'
EXPECTED_EPOCH = 6
EXPECTED_STEP = 40380
EXPECTED_SEL_P1 = 0.8950
EXPECTED_LABEL_ORDER_SHA = "a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b"
EXPECTED_EVIDENCE_ZIP_SHA = "cf27038453034504b37f2fef565c9c0eb47dbf288531797a8a8004c20d426d0e"
EXPECTED_BUNDLE_ZIP_SHA = "e25c583e76d44873e0bdeebcaa99973db8ac4763b79023afd2787902a8e65bbf"
EXPECTED_BUNDLE_MANIFEST_SHA = "8a6ebf146293e22e3a597f992924c84dd06ffc4fe5276d9978add3dfdd9fe9ce"
EXPECTED_EVIDENCE_FILE_SHAS = {'RUN_MANIFEST.json': '75744f07631f6f5977e4ef7941c1d1b8130e24fee389501b188bf18d0a0ab984', 'CONFIG_RESOLVED.json': '72cfcbc306a33c08d3333cebf5760fef15360004207c6c29aeb694dd47712510', 'dev_partition.json': 'fdeea51e0addcd9d52cb577275bf727bb958bb5a0d1bb313bdb9b6f93c5f541b', 'epoch_metrics.csv': 'd0e6116e045252ca075e75757c5ae1bfc6381164fdd431fbb87b87b3abacc122', 'train_history.json': '5af179ee65a4d388b669bfab69650d6a9b766971e8141e7e639e98e7745992f4', 'training_summary.json': '1febc6295962f4cbb837fedf8f92c59f0e898fddeb6cc39c091b52f77bd95f76', 'per_label_metrics.csv': '4c8f89ba4746558011e81d756fd87a76ab2c9ebb451ba2eafbcc3867a98b0067', 'step300_info.json': 'bb2ee0079d3a8c2fcfe6d7245d509eef80649d9bfe9e342388893c0c7cc5aa19', 'hrrm_heldout_comparison.json': 'fadde80fb59a41c54e1e2e80862b2de7ee4299c9d9d3808b4248b2412c14e6c3', 'inference_demo.json': 'a65373c3b08242ba3cc9adfc4f364d5cc0078aaadf0272ad9b874e0810d3b02a', 'neural_dev_predictions.json': '210539710c363c348b061af9387fd0c0f3ebf3ffd43686253e36717cc78aeee1'}
BERT_REPO = "google-bert/bert-base-uncased"
BERT_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
THRESHOLD = 0.5
MAX_LEN = 512
BATCH_SIZE = 8
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/LHRM_R_AUTHOR_CLIENT_INFERENCE')
assert INPUT_ROOT.exists(), 'Run this Notebook in Kaggle, with Add Input datasets attached.'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def sha256_file(path, block=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(block), b''): h.update(chunk)
    return h.hexdigest()

def unique_path(matches, purpose):
    matches = sorted(set(Path(p) for p in matches))
    if len(matches) != 1:
        raise RuntimeError(f'{purpose}: expected 1 unambiguous match, got {len(matches)}: {matches}. Attach the original files only.')
    return matches[0]

# Checkpoint may be from a user Kaggle Dataset or a restored notebook Output.
ckpt_paths = list(INPUT_ROOT.rglob('best_dev_model_only.pt'))
working_ckpt = Path('/kaggle/working/best_dev_model_only.pt')
if not ckpt_paths and working_ckpt.exists(): ckpt_paths = [working_ckpt]
CHECKPOINT_PATH = unique_path(ckpt_paths, 'Frozen best checkpoint')
if CHECKPOINT_PATH.stat().st_size < 100_000_000:
    raise RuntimeError('Checkpoint file is unexpectedly small; expected full trained BERT + CSAM model.')
print('Selected checkpoint:', CHECKPOINT_PATH)
print('Checkpoint bytes:', CHECKPOINT_PATH.stat().st_size)

# Read evidence only, not Test. Kaggle might auto-extract input ZIP uploads.
evidence_zips = [p for p in INPUT_ROOT.rglob('EVIDENCE.zip') if sha256_file(p) == EXPECTED_EVIDENCE_ZIP_SHA]
if evidence_zips:
    EVIDENCE_SOURCE = unique_path(evidence_zips, 'EVIDENCE.zip')
    with zipfile.ZipFile(EVIDENCE_SOURCE) as z:
        EVIDENCE_BYTES = {n: z.read(n) for n in EXPECTED_EVIDENCE_FILE_SHAS}
    print('[EVIDENCE] Verified original ZIP SHA:', EXPECTED_EVIDENCE_ZIP_SHA)
else:
    evidence_roots = [p.parent for p in INPUT_ROOT.rglob('RUN_MANIFEST.json')
                      if sha256_file(p) == EXPECTED_EVIDENCE_FILE_SHAS['RUN_MANIFEST.json']]
    evidence_root = unique_path(evidence_roots, 'Auto-extracted evidence root')
    EVIDENCE_SOURCE = evidence_root
    EVIDENCE_BYTES = {n:(evidence_root/n).read_bytes() for n in EXPECTED_EVIDENCE_FILE_SHAS}
    print('[EVIDENCE] Using verified Kaggle auto-extracted evidence at:', evidence_root)
for n, expected in EXPECTED_EVIDENCE_FILE_SHAS.items():
    actual = hashlib.sha256(EVIDENCE_BYTES[n]).hexdigest()
    assert actual == expected, f'Evidence hash mismatch: {n}: {actual}'
RUN_MANIFEST = json.loads(EVIDENCE_BYTES['RUN_MANIFEST.json'])
CONFIG = json.loads(EVIDENCE_BYTES['CONFIG_RESOLVED.json'])
assert RUN_MANIFEST['run_id'] == RUN_ID
assert RUN_MANIFEST['best_epoch'] == EXPECTED_EPOCH
assert RUN_MANIFEST['best_step'] == EXPECTED_STEP
assert abs(RUN_MANIFEST['best_selection_p1'] - EXPECTED_SEL_P1) < 1e-10
assert RUN_MANIFEST['label_order_hash'] == EXPECTED_LABEL_ORDER_SHA
assert CONFIG['label_order_hash'] == EXPECTED_LABEL_ORDER_SHA
assert RUN_MANIFEST['selected_checkpoint'] == CHECKPOINT_PATH.name
label_df = pd.read_csv(BytesIO(EVIDENCE_BYTES['per_label_metrics.csv']))
assert len(label_df) == 54 and label_df['index'].tolist() == list(range(54))
LABELS = label_df['label'].tolist()
assert len(set(LABELS)) == 54
assert hashlib.sha256(json.dumps(LABELS).encode()).hexdigest() == EXPECTED_LABEL_ORDER_SHA
print('[EVIDENCE] PASS: run provenance, original label order, 54-label Dev metrics')

# Code bundle: either raw zip or Kaggle auto-extracted verified sources.
bundle_zips = [p for p in INPUT_ROOT.rglob('lhrm_gpu_bundle_r_author_20261015.zip')
               if sha256_file(p) == EXPECTED_BUNDLE_ZIP_SHA]
if bundle_zips:
    BUNDLE_SOURCE = unique_path(bundle_zips, 'Bundle ZIP')
    print('[BUNDLE] Found raw verified archive:', BUNDLE_SOURCE)
else:
    manifest_candidates = [p for p in INPUT_ROOT.rglob('BUNDLE_MANIFEST.json')
                           if sha256_file(p) == EXPECTED_BUNDLE_MANIFEST_SHA]
    BUNDLE_SOURCE = unique_path(manifest_candidates, 'Auto-extracted source bundle manifest')
    print('[BUNDLE] Found verified auto-extracted manifest:', BUNDLE_SOURCE)

print('CELL 1 PASS — original evidence, 54-label order, source bundle and checkpoint located; no dataset splits read.')


In [ ]:
# CELL 2 — Verify R_AUTHOR source and load pinned BERT tokenizer/config (not full pretrained weights).
from transformers import BertTokenizer, BertConfig, BertModel

WORKING = Path('/kaggle/working')
if BUNDLE_SOURCE.suffix.lower() == '.zip':
    assert sha256_file(BUNDLE_SOURCE) == EXPECTED_BUNDLE_ZIP_SHA
    with zipfile.ZipFile(BUNDLE_SOURCE, 'r') as z:
        raw_manifest = z.read('BUNDLE_MANIFEST.json')
        assert hashlib.sha256(raw_manifest).hexdigest() == EXPECTED_BUNDLE_MANIFEST_SHA
        bundle_manifest = json.loads(raw_manifest)
        source_files = {}
        for rel_path, expected in bundle_manifest['files'].items():
            content = z.read(rel_path)
            assert hashlib.sha256(content).hexdigest() == expected, rel_path
            source_files[rel_path] = content
else:
    assert sha256_file(BUNDLE_SOURCE) == EXPECTED_BUNDLE_MANIFEST_SHA
    bundle_manifest = json.loads(BUNDLE_SOURCE.read_text())
    source_files = {}
    for rel_path, expected in bundle_manifest['files'].items():
        content = (BUNDLE_SOURCE.parent/rel_path).read_bytes()
        assert hashlib.sha256(content).hexdigest() == expected, rel_path
        source_files[rel_path] = content
assert len(source_files) == 16, f'Unexpected bundle file count: {len(source_files)}'
for rel_path, content in source_files.items():
    dest = WORKING/rel_path
    assert dest.resolve().is_relative_to(WORKING.resolve()), 'Unsafe bundle path'
    dest.parent.mkdir(parents=True,exist_ok=True)
    dest.write_bytes(content)
if str(WORKING) not in sys.path: sys.path.insert(0, str(WORKING))
from src.lhrm_baseline.model import LHRMModel
from src.lhrm_baseline.inference import predict_abstract
print('Verified and staged',len(source_files),'R_AUTHOR bundle files.')

# NOTE: best_dev_model_only.pt CONTAINS the fully fine-tuned BERT weights.
# Do NOT download another full pre-trained BERT; the original pinned config+tokenizer suffice.
TOKENIZER_DIR = WORKING/'bert_author_pinned_tokenizer_config'
TOKENIZER_DIR.mkdir(parents=True,exist_ok=True)
required = ['config.json', 'vocab.txt']
if not all((TOKENIZER_DIR/x).exists() for x in required):
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=BERT_REPO, revision=BERT_REVISION,
                          allow_patterns=['config.json','vocab.txt','tokenizer_config.json',
                                          'special_tokens_map.json','tokenizer.json'],
                          local_dir=str(TOKENIZER_DIR))
    except Exception as exc:
        raise RuntimeError('Need Kaggle Internet ON for pinned tokenizer/config; ' 
                           'alternatively attach the original official BERT tokenizer/config files.') from exc
assert all((TOKENIZER_DIR/x).exists() for x in required), 'Pinned tokenizer/config files missing.'
bert_config = BertConfig.from_pretrained(str(TOKENIZER_DIR))
assert bert_config.hidden_size == 768 and bert_config.vocab_size == 30522
TOKENIZER = BertTokenizer.from_pretrained(str(TOKENIZER_DIR))
assert len(TOKENIZER) == 30522 and TOKENIZER.pad_token_id == 0
print('CELL 2 PASS — official pinned tokenizer + architecture available; no dataset splits read.')

In [ ]:
# CELL 3 — Strictly load frozen Epoch-6 trained weights; no optimizer or training.
print('Loading best checkpoint to CPU for provenance checks...')
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert isinstance(ckpt,dict) and isinstance(ckpt.get('model_state_dict'),dict)
assert ckpt.get('run_id') == RUN_ID, f"Unexpected run ID: {ckpt.get('run_id')}"
assert ckpt.get('variant') == 'R_AUTHOR'
assert ckpt.get('epoch') == EXPECTED_EPOCH
assert ckpt.get('step') == EXPECTED_STEP
assert abs(ckpt.get('selection_P@1',-1) - EXPECTED_SEL_P1) < 1e-10
assert ckpt.get('selection_subset') == '400_docs'
assert abs(ckpt.get('full_dev_P@1', -1) - 0.8760) < 1e-10
assert 'optimizer_state_dict' not in ckpt, 'Unexpected training checkpoint, use best_dev_model_only.pt'
sd = ckpt['model_state_dict']
assert tuple(sd['csam.label_emb'].shape) == (54, 768)
assert tuple(sd['csam.classifier.weight'].shape) == (54,1536)
print('Verified frozen checkpoint metadata: Epoch 6, Step 40380, Selection P@1=0.8950.')
assert sha256_file(CHECKPOINT_PATH) == EXPECTED_CHECKPOINT_SHA, 'Wrong checkpoint bytes — STOP'
print('Checkpoint SHA-256 VERIFIED:', EXPECTED_CHECKPOINT_SHA)

# Build identical model architecture. The dummy label_emb is FULLY OVERWRITTEN by trained state_dict.
# This avoids expensive and unnecessary BERT label embedding re-computation.
BERT = BertModel(bert_config)
MODEL = LHRMModel(bert_model=BERT, num_labels=len(LABELS),
                  variant='R_AUTHOR', label_emb=torch.zeros(54,768))
MODEL.load_state_dict(sd, strict=True)
MODEL = MODEL.to(DEVICE).eval()
for p in MODEL.parameters(): p.requires_grad_(False)
assert all(not p.requires_grad for p in MODEL.parameters())
with torch.inference_mode():
    enc = TOKENIZER('Text classification sanity check.', return_tensors='pt',
                    padding='max_length', truncation=True, max_length=MAX_LEN)
    logits, _ = MODEL(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
assert tuple(logits.shape) == (1, 54) and torch.isfinite(logits).all()
print('CELL 3 PASS — verified trained weights loaded strictly, inference-only.')

In [ ]:
# CELL 4 — Verify that the copied client model matches 3 original inference records.
# No Train/Dev/Test inputs are needed; all 3 texts come from original demonstration evidence.
SMOKE_TEXTS = [
    'We present a novel neural architecture for multi-label text classification using contextual similarity attention and label embeddings. Our hierarchical ranking approach learns cross-label dependencies and optimizes ranking metrics directly.',
    'Quantum phase transitions in disordered spin systems are investigated using numerical Monte Carlo simulations. We examine the low-temperature critical exponents and spatial correlation functions across the disordered phase boundary.',
    'Efficient algorithms for convex optimization in high-dimensional machine learning problems. We analyze stochastic gradient methods with adaptive step-sizes and prove linear convergence under strong convexity assumptions.'
]
old_demo = json.loads(EVIDENCE_BYTES['inference_demo.json'])
assert len(old_demo) == len(SMOKE_TEXTS) == 3
for i,(text,reference) in enumerate(zip(SMOKE_TEXTS,old_demo),1):
    with torch.inference_mode():
        out = predict_abstract(text=text,model=MODEL,tokenizer=TOKENIZER,
                               label_order=LABELS,device=DEVICE,hrrm_model=None,top_k=5)
    actual,expected = out['neural_top_k'],reference['neural_top_k']
    assert [x['label'] for x in actual] == [x['label'] for x in expected], f'Smoke {i}: top5 mismatch'
    max_abs_diff = max(abs(a['score']-e['score']) for a,e in zip(actual,expected))
    assert max_abs_diff < 0.002, f'Smoke {i}: model/tokenizer score mismatch {max_abs_diff}'
    print(f'[{i}/3] PASS | top1={actual[0]["label"]} | max score diff={max_abs_diff:.7f}')
SMOKE_PASSED = True
print('CELL 4 PASS — all three client inference replays match original training evidence. Ready for new abstracts.')

In [ ]:
# CELL 5 — EDIT THE TEXT BELOW and run this cell to classify ONE new abstract.
# NOTE: Example text is purely a demonstration, NOT ground truth or a validated label mapping.
# Truncation: max 512 BERT tokens; sigmoid scores are NOT calibrated probabilities.

CLIENT_ABSTRACT = """We study representation learning for text classification using transformer
encoders and attention. We compare methods on a collection of scientific abstracts."""
TOP_K = 5  # Show top five ranked codes from the original AAPD dataset.

from IPython.display import display, Markdown

def classify_client_abstract(text, top_k=5, show_table=True):
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Please enter a non-empty English scientific abstract.')
    if len(text) > 100_000:
        raise ValueError('Input too long. Paste a scientific abstract, not a full paper.')
    if not isinstance(top_k, int) or not (1 <= top_k <= len(LABELS)):
        raise ValueError('top_k must be an integer between 1 and 54.')
    if not SMOKE_PASSED:
        raise RuntimeError('Run Cell 4 and verify the original inference replay first.')
    token_count = len(TOKENIZER.encode(text, add_special_tokens=True, truncation=False))
    with torch.inference_mode():
        outcome = predict_abstract(
            text=text.strip(), model=MODEL, tokenizer=TOKENIZER,
            label_order=LABELS, device=DEVICE,
            hrrm_model=None, top_k=top_k, max_len=MAX_LEN,
        )
    rows = [
        {'Rank': int(r['rank']), 'Raw AAPD Label': str(r['label']),
         'Sigmoid score (0–1)': float(r['score']),
         'Above 0.5': 'Yes' if float(r['score']) >= THRESHOLD else 'No'}
        for r in outcome['neural_top_k']
    ]
    result_df = pd.DataFrame(rows)
    if show_table:
        display(Markdown('### Ranked model predictions (not verified topic names)'))
        display(result_df.style.format({'Sigmoid score (0–1)': '{:.4f}'}))
        if token_count > MAX_LEN:
            print(f'WARNING: Input is {token_count} tokens; only first {MAX_LEN} tokens were processed.')
        display(Markdown(
            '**Interpretation:** Original AAPD column codes; mapping of label names to true '
            'topics remains UNVERIFIED. Scores are uncalibrated sigmoid outputs, not factual '
            'confidence guarantees. The 0.5 column is a fixed display cutoff, not tuned for the input.'
        ))
    return {'dataframe': result_df, 'token_count': token_count, 'raw': outcome}

client_result = classify_client_abstract(CLIENT_ABSTRACT, top_k=TOP_K)
print('CELL 5 PASS — client text classified using frozen trained weights (no training).')


In [ ]:
# CELL 6 — OPTIONAL interactive text box. Requires ipywidgets (usually available on Kaggle).
# If widgets do not render, use Cell 5 and edit CLIENT_ABSTRACT instead.

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print('ipywidgets is unavailable here. Use Cell 5 to paste your abstract and run inference.')
else:
    input_box = widgets.Textarea(
        value='', placeholder='Paste an English scientific abstract here...',
        description='Abstract:', layout=widgets.Layout(width='100%', height='170px'),
        style={'description_width': 'initial'},
    )
    predict_button = widgets.Button(description='Predict Top 5', button_style='primary', icon='search')
    result_box = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

    def on_predict_clicked(_):
        with result_box:
            clear_output(wait=True)
            try:
                _ = classify_client_abstract(input_box.value, top_k=5, show_table=True)
            except Exception as exc:
                print('Cannot classify:', type(exc).__name__, str(exc))

    predict_button.on_click(on_predict_clicked)
    display(widgets.VBox([
        widgets.HTML('<h3>LHRM — Client Abstract Classifier</h3><p>'
                     'Frozen Epoch 6 · original AAPD label codes · uncalibrated scores</p>'),
        input_box, predict_button, result_box,
        widgets.HTML('<small>Warning: semantic label mapping is unresolved. '
                     'This notebook does not train or tune any model.</small>'),
    ]))


## Session and delivery notes

- **No AAPD dataset is needed here.** There is no `train.csv`, `dev.csv` or `test.csv` access in this notebook.
- Save the verified trained checkpoint and evidence separately (private data input); this notebook does **not** embed or distribute the binary weights.
- For client delivery, retain the frozen **Test** report separately (Test P@1 **0.8490**, Micro-F1 **0.727787**, Macro-F1 **0.555253** on 1,000 docs/54 labels); do not recompute/adjust these numbers based on client demos.
- HRRM is **not enabled** in this client demo because the held-out Dev comparison found no improvement (P@1 **0.8700 → 0.8650**). HRRM was **not evaluated on Test**.
- To share your own notebook, **clear its cell outputs** to prevent accidental disclosure of client text.
